In [ ]:
%pip install nltk
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from wordcloud import WordCloud

from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/lilswapnil/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
pain_points = '../output/cleaned_challenges.csv'
expectations = '../output/cleaned_expectations.csv'

df_pain_points = pd.read_csv(pain_points)
df_expectations = pd.read_csv(expectations)

In [ ]:
df_pain_points["VADER"] = df_pain_points["Pain Point"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

df_pain_points.head()

,Focus Group,Pain Point,processed_pain_points,label,vader_sentiment,VADER
0,CPO Central Permit Office,Split Permit & Code Systems: Permit and Code E...,split permit code system permit code enforceme...,neutral,neutral,0.0000
1,CPO Central Permit Office,Scattered Information: Property and permit inf...,scattered information property permit informat...,neutral,neutral,0.0000
2,CPO Central Permit Office,Research Complexity: Users must search multipl...,research complexity user must search multiple ...,neutral,neutral,0.0000
3,CPO Central Permit Office,Difficult Information Retrieval: Finding relev...,difficult information retrieval finding releva...,neutral,neutral,0.0000
4,CPO Central Permit Office,Limited IPS-Camino Integration: Information do...,limited integration information flow seamlessl...,negative,negative,-0.2263


In [ ]:
df_expectations["VADER"] = df_expectations["Expectation"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

df_expectations.head()

,Focus Group,Expectation,processed_expectations,vader_sentiment,VADER
0,CPO Central Permit Office,Unified Permit & Code Platform: Create a seaml...,unified permit code platform create seamless c...,positive,0.5719
1,CPO Central Permit Office,Shared Case Visibility: Allow permit reviewers...,shared case visibility allow permit reviewer c...,positive,0.5106
2,CPO Central Permit Office,Automatic Workflow Triggers: Permit and violat...,automatic workflow trigger permit violation ac...,negative,-0.4939
3,CPO Central Permit Office,"Integrated Property Records: Display permits, ...",integrated property record display permit viol...,negative,-0.5267
4,CPO Central Permit Office,Single Property View: Consolidate all property...,single property view consolidate property info...,neutral,0.0000


In [ ]:
negative_words = {

    "manual":3,
    "missing":3,
    "difficult":3,
    "slow":2,
    "multiple":2,
    "duplicate":2,
    "limited":2,
    "cannot":3,
    "can't":3,
    "hard":2,
    "confusing":2,
    "problem":2,
    "issue":2,
    "fragmented":3,
    "scattered":3,
    "research":1,
    "paper":2,
    "search":1,
    "delay":2,
    "dependency":2,
    "outdated":3,
    "incomplete":3,
    "legacy":2,
    "repetitive":2

}

In [ ]:
def business_sentiment(text):

    score = 0

    text = text.lower()

    for word, value in negative_words.items():

        if word in text:

            score -= value

    return score


df_pain_points["Business Score"] = df_pain_points["Pain Point"].apply(business_sentiment)

In [ ]:
positive_words = {

    "improved":3,
    "improve":3,
    "better":2,
    "automated":3,
    "automation":3,
    "integrated":3,
    "integration":3,
    "single":2,
    "streamlined":3,
    "efficient":2,
    "dashboard":2,
    "visibility":2,
    "tracking":2,
    "centralized":3,
    "search":2,
    "mobile":2,
    "reporting":2,
    "workflow":2,
    "notifications":2,
    "communication":2,
    "collaboration":2,
    "history":2,
    "records":1,
    "gis":2,
    "real-time":3,
    "access":2,
    "sharing":2,
    "user-friendly":3,
    "simplified":3,
    "reduced":1,
    "faster":2

}

In [ ]:
def business_sentiment(text):

    score = 0

    text = text.lower()

    for word, value in positive_words.items():

        if word in text:

            score += value

    return score


df_expectations["Business Score"] = df_expectations["Expectation"].apply(business_sentiment)

In [ ]:
stakeholder_scores = (
    df_pain_points
    .groupby("Focus Group")
    .agg(
        BusinessScore=("Business Score","sum"),
        AverageBusinessScore=("Business Score","mean"),
        Vader=("VADER","mean"),
        PainPoints=("Pain Point","count")
    )
    .sort_values("BusinessScore")
)

stakeholder_scores

,BusinessScore,AverageBusinessScore,Vader,PainPoints
Focus Group,,,,
DOCE CommercialPermitElectrical Inspectors,-73,-2.354839,-0.160897,31
NBD NBD Internal,-70,-2.800000,0.066208,25
DOCE Supervisors,-62,-2.296296,-0.183207,27
DOCE Fire Prevention Bureau,-60,-2.400000,-0.144788,25
DOCE Housing Inspectors,-58,-1.933333,-0.116633,30
DOCE Zoning,-55,-5.000000,-0.036300,11
CPO CPO Co-Ordinator,-51,-2.125000,-0.044038,24
DOCE Building Inspectors,-46,-1.769231,-0.157681,26
CPO Central Permit Office,-45,-2.250000,-0.025915,20


In [ ]:
stakeholder_scores = (
    df_expectations
    .groupby("Focus Group")
    .agg(
        BusinessScore=("Business Score","sum"),
        AverageBusinessScore=("Business Score","mean"),
        Vader=("VADER","mean"),
        PainPoints=("Expectation","count")
    )
    .sort_values("BusinessScore")
)

stakeholder_scores

,BusinessScore,AverageBusinessScore,Vader,PainPoints
Focus Group,,,,
DOCE Building Inspectors,-112,-4.000000,0.149271,28
DOCE Office Manager,-111,-3.171429,0.280937,35
CPO Central Permit Office,-106,-4.240000,0.258588,25
DOCE CommercialPermitElectrical Inspectors,-103,-3.678571,0.241386,28
DOCE Supervisors,-103,-3.322581,0.223742,31
DOCE Housing Inspectors,-99,-3.000000,0.234127,33
DOCE Fire Prevention Bureau,-97,-4.041667,0.108250,24
BAA BAA Supervisors,-65,-5.909091,0.212055,11
DOCE Zoning,-60,-6.666667,0.388156,9


In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]

    lemmas = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    return ' '.join(lemmas)

df_pain_points['processed_pain_points'] = df_pain_points['Pain Point'].apply(preprocess_text)
print(df_pain_points[['Focus Group', 'Pain Point', 'processed_pain_points']].head(10))

In [ ]:
df_expectations['processed_expectations'] = df_expectations['Expectation'].apply(preprocess_text)
print(df_expectations[['Focus Group', 'Expectation', 'processed_expectations']].head(10))

In [ ]:
df_pain_points.shape, df_expectations.shape

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter

# Rebuild processed_content here if the preprocessing cell has not been run.
if 'processed_content' not in df_pain_points.columns:
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        tokens = word_tokenize(str(text))
        filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]
        lemmas = [lemmatizer.lemmatize(token) for token in filtered_tokens]
        return ' '.join(lemmas)

    df_pain_points['processed_pain_points'] = df_pain_points['Pain Point'].apply(preprocess_text)

if 'processed_expectations' not in df_expectations.columns:
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        tokens = word_tokenize(str(text))
        filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]
        lemmas = [lemmatizer.lemmatize(token) for token in filtered_tokens]
        return ' '.join(lemmas)

    df_expectations['processed_expectations'] = df_expectations['Expectation'].apply(preprocess_text)


# def get_category(text):
#     tokens = str(text).split()
#     if not tokens:
#         return 'unknown'
#     return Counter(tokens).most_common(1)[0][0]

sia = SentimentIntensityAnalyzer()

def get_sentiment_label(text):
    text = str(text)
    compound_score = sia.polarity_scores(text)['compound']
    token_count = len([token for token in text.split() if token.strip()])

    if token_count == 1 and -0.05 < compound_score < 0.05:
        return 'neutral'
    if compound_score >= 0.05:
        return 'positive'
    if compound_score <= -0.05:
        return 'negative'
    return 'neutral'

# Build a label column from the cleaned text so the model has a target to learn.
df_pain_points['label'] = df_pain_points['processed_pain_points'].apply(get_sentiment_label)
df_pain_points = df_pain_points[df_pain_points['label'].isin(['positive', 'negative', 'neutral'])].copy()

# Create a category from the dominant word in the cleaned content.
# df_pain_points['category'] = df_pain_points['processed_pain_points'].apply(get_category)
# df_expectations['category'] = df_expectations['processed_expectations'].apply(get_category)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df_pain_points['processed_pain_points'])
y = df_pain_points['label'].map({'positive': 1, 'negative': 0, 'neutral': 2}).values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
# print(df_pain_points[['Pain Point', 'processed_pain_points', 'category', 'label']].head(10))
# print(df_expectations[['Focus Group', 'Expectation', 'processed_expectations', 'category']].head(10))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[
            'Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
df_pain_points.sample(5)

In [ ]:
def predict_sentiment(text):
    processed = preprocess_text(text)
    vectorized = vectorizer.transform([processed])
    pred = model.predict(vectorized)[0]
    return {0: 'negative', 1: 'positive', 2: 'neutral'}.get(pred, 'neutral')


for text in df_pain_points['Pain Point'].sample(5, random_state=42):
    print(f"Sentence: {text} => Sentiment: {predict_sentiment(text)}")

In [ ]:
import joblib

joblib.dump(model, '../models/sentiment_nb_model.joblib')
joblib.dump(vectorizer, '../models/count_vectorizer.joblib')

In [ ]:
df_pain_points.to_csv('../output/pain_points.csv', index=False, encoding='utf-8-sig')
df_expectations.to_csv('../output/expectations.csv', index=False, encoding='utf-8-sig')